In [0]:
-- req 1
WITH match_bracket AS (
    -- Get all knockout matches with bracket structure
    SELECT 
        match_id,
        stage,
        bracket_position,
        feeds_into_position,
        bracket_half,
        
        -- Home team (handles TBD logic)
        home_team_name,
        COALESCE(home_display_name, home_team_name, 'TBD') AS home_display_name,
        home_team_logo,
        home_score,
        
        -- Away team (handles TBD logic)
        away_team_name,
        COALESCE(away_display_name, away_team_name, 'TBD') AS away_display_name,
        away_team_logo,
        away_score,
        
        -- Match outcome
        winner_team_name,
        winner_team_logo,
        
        -- Match metadata
        match_status,
        match_datetime_utc,
        match_date_local,
        stadium_name,
        stadium_city,
        actual_country,
        is_finished,
        is_live
        
    FROM singhal.fifa_worldcup_gold.gold_fact_match_schedule
    WHERE is_knockout = TRUE
),

home_goals AS (
    -- Get goals scored by home team in each match
    SELECT
        g.match_id,
        COLLECT_LIST(
            STRUCT(
                g.scorer_name,
                g.minute,
                g.is_penalty,
                g.goal_number_in_match
            )
        ) AS goals
    FROM singhal.fifa_worldcup_gold.gold_team_goals g
    INNER JOIN match_bracket mb ON g.match_id = mb.match_id AND g.team_name = mb.home_team_name
    GROUP BY g.match_id
),

away_goals AS (
    -- Get goals scored by away team in each match
    SELECT
        g.match_id,
        COLLECT_LIST(
            STRUCT(
                g.scorer_name,
                g.minute,
                g.is_penalty,
                g.goal_number_in_match
            )
        ) AS goals
    FROM singhal.fifa_worldcup_gold.gold_team_goals g
    INNER JOIN match_bracket mb ON g.match_id = mb.match_id AND g.team_name = mb.away_team_name
    GROUP BY g.match_id
)

-- Final comprehensive query
SELECT
    -- Bracket structure
    mb.match_id,
    mb.stage,
    mb.bracket_position,              -- e.g., "QF1", "SF1", "FIN"
    mb.feeds_into_position,           -- e.g., QF1 → SF1
    mb.bracket_half,                  -- "Top" or "Bottom"
    
    -- Home team
    mb.home_team_name,
    mb.home_display_name,             -- Shows "Winner of QF1" or "TBD" for unfilled
    mb.home_team_logo,                -- 🇦🇷 Team flag
    mb.home_score,
    
    -- Home team goals (formatted with timing)
    CASE 
        WHEN mb.is_finished = TRUE AND hg.goals IS NOT NULL
        THEN TRANSFORM(
            hg.goals,
            g -> CONCAT(
                g.scorer_name, ' (', g.minute, ')',
                CASE WHEN g.is_penalty THEN ' (PEN)' ELSE '' END
            )
        )
        ELSE ARRAY('NA')  -- Match not completed yet
    END AS home_goals_detail,
    
    -- Away team
    mb.away_team_name,
    mb.away_display_name,             -- Shows "Winner of QF3" or "TBD" for unfilled
    mb.away_team_logo,                -- 🇫🇷 Team flag
    mb.away_score,
    
    -- Away team goals (formatted with timing)
    CASE 
        WHEN mb.is_finished = TRUE AND ag.goals IS NOT NULL
        THEN TRANSFORM(
            ag.goals,
            g -> CONCAT(
                g.scorer_name, ' (', g.minute, ')',
                CASE WHEN g.is_penalty THEN ' (PEN)' ELSE '' END
            )
        )
        ELSE ARRAY('NA')  -- Match not completed yet
    END AS away_goals_detail,
    
    -- Match outcome
    mb.winner_team_name,
    mb.winner_team_logo,
    
    -- Match info
    mb.match_status,                  -- 'Finished', 'Live', 'Upcoming'
    mb.match_datetime_utc,            -- UTC time
    mb.match_date_local,              -- Local timezone display
    mb.stadium_name,                  -- MetLife Stadium
    mb.stadium_city,                  -- New Jersey
    mb.actual_country,                -- United States
    mb.is_finished,
    mb.is_live

FROM match_bracket mb
LEFT JOIN home_goals hg ON mb.match_id = hg.match_id
LEFT JOIN away_goals ag ON mb.match_id = ag.match_id

ORDER BY 
    CASE mb.stage
        WHEN 'Round of 32' THEN 1
        WHEN 'Round of 16' THEN 2
        WHEN 'Quarter Final' THEN 3
        WHEN 'Semi Final' THEN 4
        WHEN 'Third Place' THEN 5
        WHEN 'Final' THEN 6
    END,
    mb.bracket_position;

In [0]:
-- req 2
SELECT 
    rank,
    player_id,
    player_name,
    player_logo,
    team_id,
    team_name,
    team_logo,
    goals_scored,
    assists,
    matches_played,
    minutes_played,
    rating_0_to_10,
    goals_percentile,      -- For spider chart
    assists_percentile,    -- For spider chart
    minutes_percentile,    -- For spider chart
    matches_percentile,    -- For spider chart
    most_goals_against_team_name,
    most_goals_against_team_count,
    golden_boot_rank,
    is_top_3,
    goals_behind_leader
FROM singhal.fifa_worldcup_gold.gold_player_leaderboard
ORDER BY rank
LIMIT 10;

In [0]:
-- req 3
SELECT 
    match_id,
    match_status,  -- 'Live' or 'Upcoming'
    match_datetime_utc,
    match_date_local,
    home_team_id,
    home_team_name,
    home_team_logo,
    home_top_scorer_name,
    home_top_scorer_goals,
    home_score,
    away_team_id,
    away_team_name,
    away_team_logo,
    away_top_scorer_name,
    away_top_scorer_goals,
    away_score,
    stage,
    group_name,  -- NULL for knockout matches
    is_knockout,
    stadium_name,
    stadium_city,
    actual_country,
    is_live,
    minutes_elapsed,  -- For live matches
    hours_until_kickoff  -- For upcoming matches
FROM singhal.fifa_worldcup_gold.gold_fact_match_schedule
WHERE match_status IN ('Live', 'Upcoming')
  AND is_finished = FALSE
  AND home_team_name IS NOT NULL  -- Exclude TBD matches
  AND away_team_name IS NOT NULL  -- Exclude "Winner of X" matches
ORDER BY match_datetime_utc ASC;

In [0]:
-- req 4
SELECT 
    m.match_id,
    
    -- Team information
    m.home_team_name AS team_a_name,
    m.home_team_logo AS team_a_logo,
    m.away_team_name AS team_b_name,
    m.away_team_logo AS team_b_logo,
    
    -- Score information
    m.home_score,
    m.away_score,
    
    -- Winner information
    m.winner_team_name,
    m.winner_team_logo,
    
    -- Penalty shootout detection
    -- If knockout match ended in a draw but has a winner, it was decided by penalties
    CASE 
        WHEN m.is_knockout = TRUE 
         AND m.home_score = m.away_score 
         AND m.winner_team_name IS NOT NULL 
        THEN TRUE 
        ELSE FALSE 
    END AS is_penalty_shootout,
    
    -- Match timing
    m.match_datetime_utc,
    m.match_date_local,
    
    -- Location
    m.stadium_city,
    m.stadium_name,
    m.actual_country,
    
    -- Tournament context
    m.stage,
    m.group_name,  -- NULL for knockout matches
    m.is_knockout
    
FROM singhal.fifa_worldcup_gold.gold_fact_match_schedule m
WHERE m.is_finished = TRUE
ORDER BY m.match_datetime_utc DESC;

In [0]:
-- req 5a
SELECT 
    team_name,
    team_logo,
    total_points,
    total_wins,
    total_losses,
    total_draws,
    goals_for,
    goals_against,
    goal_difference,
    clean_sheets,
    group_name,  -- e.g., "A", "B", "C" or NULL for knockout stage
    current_stage,  -- e.g., "Group Stage", "Quarter Final", "Semi Final"
    qualification_status,  -- 'Qualified', 'Disqualified', 'In Progress'
    rank_overall  -- Overall tournament ranking
FROM singhal.fifa_worldcup_gold.gold_fact_team_performance
ORDER BY total_points DESC, goal_difference DESC, goals_for DESC;

In [0]:
-- req 5b: All match history (dashboard will filter by clicked team)
SELECT 
    m.match_id,
    m.match_datetime_utc,
    m.match_date_local,
    m.stage,
    
    -- Home team info
    m.home_team_name,
    m.home_team_logo,
    m.home_score,
    
    -- Away team info
    m.away_team_name,
    m.away_team_logo,
    m.away_score,
    
    -- Winner info
    m.winner_team_name,
    m.winner_team_logo,
    
    -- Penalty shootout detection
    CASE 
        WHEN m.is_knockout = TRUE 
         AND m.home_score = m.away_score 
         AND m.winner_team_name IS NOT NULL 
        THEN TRUE 
        ELSE FALSE 
    END AS is_penalty_shootout,
    
    m.stadium_name,
    m.stadium_city
    
FROM singhal.fifa_worldcup_gold.gold_fact_match_schedule m
WHERE m.is_finished = TRUE
ORDER BY m.match_datetime_utc ASC;

In [0]:
-- 6 req
SELECT 
    match_sequence,  -- X-axis: Match number (1, 2, 3...)
    player_id,
    player_name,
    player_logo,  -- Player headshot for legend
    team_id,
    team_name,
    team_logo,
    goals_cumulative,  -- Y-axis: Cumulative goals
    assists_cumulative,
    minutes_cumulative,
    rank_at_match,  -- 1, 2, or 3
    medal,  -- 🥇, 🥈, or 🥉
    is_current_top_3  -- TRUE only for latest match (filter for current standings)
FROM singhal.fifa_worldcup_gold.gold_golden_boot_race
WHERE rank_at_match <= 3  -- Only top 3 players
ORDER BY match_sequence, rank_at_match;

In [0]:
-- req 7
SELECT 
    tournament_name,        -- e.g., "FIFA World Cup 2026"
    total_matches,          -- Total matches in tournament
    completed_matches,      -- Matches finished
    remaining_matches,      -- Matches yet to be played
    total_goals,            -- Total goals scored
    avg_goals_per_match,    -- Average goals per completed match
    current_round,          -- e.g., "Group Stage", "Quarter Finals", "Semi Finals"
    teams_remaining,        -- Number of teams still in competition
    top_scorer_name,        -- Current leading scorer
    top_scorer_goals        -- Goals by top scorer
FROM singhal.fifa_worldcup_gold.gold_tournament_summary;

In [0]:
%python
import json
import os

# Target directory
output_dir = "/Workspace/Users/abhisheksinghal861@gmail.com/Data-Pipelines/dashboard/data"
                                                                                                
# Create directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

print(f"✅ Export directory ready: {output_dir}")

In [0]:
%python
# Dictionary of all SQL queries
queries = {
    "req_1": """
        WITH match_bracket AS (
    -- Get all knockout matches with bracket structure
    SELECT 
        match_id,
        stage,
        bracket_position,
        feeds_into_position,
        bracket_half,
        
        -- Home team (handles TBD logic)
        home_team_name,
        COALESCE(home_display_name, home_team_name, 'TBD') AS home_display_name,
        home_team_logo,
        home_score,
        
        -- Away team (handles TBD logic)
        away_team_name,
        COALESCE(away_display_name, away_team_name, 'TBD') AS away_display_name,
        away_team_logo,
        away_score,
        
        -- Match outcome
        winner_team_name,
        winner_team_logo,
        
        -- Match metadata
        match_status,
        match_datetime_utc,
        match_date_local,
        stadium_name,
        stadium_city,
        actual_country,
        is_finished,
        is_live
        
    FROM singhal.fifa_worldcup_gold.gold_fact_match_schedule
    WHERE is_knockout = TRUE
),

home_goals AS (
    -- Get goals scored by home team in each match
    SELECT
        g.match_id,
        COLLECT_LIST(
            STRUCT(
                g.scorer_name,
                g.minute,
                g.is_penalty,
                g.goal_number_in_match
            )
        ) AS goals
    FROM singhal.fifa_worldcup_gold.gold_team_goals g
    INNER JOIN match_bracket mb ON g.match_id = mb.match_id AND g.team_name = mb.home_team_name
    GROUP BY g.match_id
),

away_goals AS (
    -- Get goals scored by away team in each match
    SELECT
        g.match_id,
        COLLECT_LIST(
            STRUCT(
                g.scorer_name,
                g.minute,
                g.is_penalty,
                g.goal_number_in_match
            )
        ) AS goals
    FROM singhal.fifa_worldcup_gold.gold_team_goals g
    INNER JOIN match_bracket mb ON g.match_id = mb.match_id AND g.team_name = mb.away_team_name
    GROUP BY g.match_id
)

-- Final comprehensive query
SELECT
    -- Bracket structure
    mb.match_id,
    mb.stage,
    mb.bracket_position,              -- e.g., "QF1", "SF1", "FIN"
    mb.feeds_into_position,           -- e.g., QF1 → SF1
    mb.bracket_half,                  -- "Top" or "Bottom"
    
    -- Home team
    mb.home_team_name,
    mb.home_display_name,             -- Shows "Winner of QF1" or "TBD" for unfilled
    mb.home_team_logo,                -- 🇦🇷 Team flag
    mb.home_score,
    
    -- Home team goals (formatted with timing)
    CASE 
        WHEN mb.is_finished = TRUE AND hg.goals IS NOT NULL
        THEN TRANSFORM(
            hg.goals,
            g -> CONCAT(
                g.scorer_name, ' (', g.minute, ')',
                CASE WHEN g.is_penalty THEN ' (PEN)' ELSE '' END
            )
        )
        ELSE ARRAY('NA')  -- Match not completed yet
    END AS home_goals_detail,
    
    -- Away team
    mb.away_team_name,
    mb.away_display_name,             -- Shows "Winner of QF3" or "TBD" for unfilled
    mb.away_team_logo,                -- 🇫🇷 Team flag
    mb.away_score,
    
    -- Away team goals (formatted with timing)
    CASE 
        WHEN mb.is_finished = TRUE AND ag.goals IS NOT NULL
        THEN TRANSFORM(
            ag.goals,
            g -> CONCAT(
                g.scorer_name, ' (', g.minute, ')',
                CASE WHEN g.is_penalty THEN ' (PEN)' ELSE '' END
            )
        )
        ELSE ARRAY('NA')  -- Match not completed yet
    END AS away_goals_detail,
    
    -- Match outcome
    mb.winner_team_name,
    mb.winner_team_logo,
    
    -- Match info
    mb.match_status,                  -- 'Finished', 'Live', 'Upcoming'
    mb.match_datetime_utc,            -- UTC time
    mb.match_date_local,              -- Local timezone display
    mb.stadium_name,                  -- MetLife Stadium
    mb.stadium_city,                  -- New Jersey
    mb.actual_country,                -- United States
    mb.is_finished,
    mb.is_live

FROM match_bracket mb
LEFT JOIN home_goals hg ON mb.match_id = hg.match_id
LEFT JOIN away_goals ag ON mb.match_id = ag.match_id

ORDER BY 
    CASE mb.stage
        WHEN 'Round of 32' THEN 1
        WHEN 'Round of 16' THEN 2
        WHEN 'Quarter Final' THEN 3
        WHEN 'Semi Final' THEN 4
        WHEN 'Third Place' THEN 5
        WHEN 'Final' THEN 6
    END,
    mb.bracket_position;
    """,
    
    "req_2": """
        SELECT 
    rank,
    player_id,
    player_name,
    player_logo,
    team_id,
    team_name,
    team_logo,
    goals_scored,
    assists,
    matches_played,
    minutes_played,
    rating_0_to_10,
    goals_percentile,      -- For spider chart
    assists_percentile,    -- For spider chart
    minutes_percentile,    -- For spider chart
    matches_percentile,    -- For spider chart
    most_goals_against_team_name,
    most_goals_against_team_count,
    golden_boot_rank,
    is_top_3,
    goals_behind_leader
FROM singhal.fifa_worldcup_gold.gold_player_leaderboard
ORDER BY rank
LIMIT 10;
    """,
    
    "req_3": """
        SELECT 
    match_id,
    match_status,  -- 'Live' or 'Upcoming'
    match_datetime_utc,
    match_date_local,
    home_team_id,
    home_team_name,
    home_team_logo,
    home_top_scorer_name,
    home_top_scorer_goals,
    home_score,
    away_team_id,
    away_team_name,
    away_team_logo,
    away_top_scorer_name,
    away_top_scorer_goals,
    away_score,
    stage,
    group_name,  -- NULL for knockout matches
    is_knockout,
    stadium_name,
    stadium_city,
    actual_country,
    is_live,
    minutes_elapsed,  -- For live matches
    hours_until_kickoff  -- For upcoming matches
FROM singhal.fifa_worldcup_gold.gold_fact_match_schedule
WHERE match_status IN ('Live', 'Upcoming')
  AND is_finished = FALSE
  AND home_team_name IS NOT NULL  -- Exclude TBD matches
  AND away_team_name IS NOT NULL  -- Exclude "Winner of X" matches
ORDER BY match_datetime_utc ASC;
    """,
    
    "req_4": """
        SELECT 
    m.match_id,
    
    -- Team information
    m.home_team_name AS team_a_name,
    m.home_team_logo AS team_a_logo,
    m.away_team_name AS team_b_name,
    m.away_team_logo AS team_b_logo,
    
    -- Score information
    m.home_score,
    m.away_score,
    
    -- Winner information
    m.winner_team_name,
    m.winner_team_logo,
    
    -- Penalty shootout detection
    -- If knockout match ended in a draw but has a winner, it was decided by penalties
    CASE 
        WHEN m.is_knockout = TRUE 
         AND m.home_score = m.away_score 
         AND m.winner_team_name IS NOT NULL 
        THEN TRUE 
        ELSE FALSE 
    END AS is_penalty_shootout,
    
    -- Match timing
    m.match_datetime_utc,
    m.match_date_local,
    
    -- Location
    m.stadium_city,
    m.stadium_name,
    m.actual_country,
    
    -- Tournament context
    m.stage,
    m.group_name,  -- NULL for knockout matches
    m.is_knockout
    
FROM singhal.fifa_worldcup_gold.gold_fact_match_schedule m
WHERE m.is_finished = TRUE
ORDER BY m.match_datetime_utc DESC;
    """,
    
    "req_5a": """
        SELECT 
    team_name,
    team_logo,
    total_points,
    total_wins,
    total_losses,
    total_draws,
    goals_for,
    goals_against,
    goal_difference,
    clean_sheets,
    group_name,  -- e.g., "A", "B", "C" or NULL for knockout stage
    current_stage,  -- e.g., "Group Stage", "Quarter Final", "Semi Final"
    qualification_status,  -- 'Qualified', 'Disqualified', 'In Progress'
    rank_overall  -- Overall tournament ranking
FROM singhal.fifa_worldcup_gold.gold_fact_team_performance
ORDER BY total_points DESC, goal_difference DESC, goals_for DESC;
    """,
    
    "req_5b": """
        SELECT 
    m.match_id,
    m.match_datetime_utc,
    m.match_date_local,
    m.stage,
    
    -- Home team info
    m.home_team_name,
    m.home_team_logo,
    m.home_score,
    
    -- Away team info
    m.away_team_name,
    m.away_team_logo,
    m.away_score,
    
    -- Winner info
    m.winner_team_name,
    m.winner_team_logo,
    
    -- Penalty shootout detection
    CASE 
        WHEN m.is_knockout = TRUE 
         AND m.home_score = m.away_score 
         AND m.winner_team_name IS NOT NULL 
        THEN TRUE 
        ELSE FALSE 
    END AS is_penalty_shootout,
    
    m.stadium_name,
    m.stadium_city
    
FROM singhal.fifa_worldcup_gold.gold_fact_match_schedule m
WHERE m.is_finished = TRUE
ORDER BY m.match_datetime_utc ASC;
    """,
    
    "req_6": """
       SELECT 
    match_sequence,  -- X-axis: Match number (1, 2, 3...)
    player_id,
    player_name,
    player_logo,  -- Player headshot for legend
    team_id,
    team_name,
    team_logo,
    goals_cumulative,  -- Y-axis: Cumulative goals
    assists_cumulative,
    minutes_cumulative,
    rank_at_match,  -- 1, 2, or 3
    medal,  -- 🥇, 🥈, or 🥉
    is_current_top_3  -- TRUE only for latest match (filter for current standings)
FROM singhal.fifa_worldcup_gold.gold_golden_boot_race
WHERE rank_at_match <= 3  -- Only top 3 players
ORDER BY match_sequence, rank_at_match;
    """,
    
    "req_7": """
        SELECT 
    tournament_name,        -- e.g., "FIFA World Cup 2026"
    total_matches,          -- Total matches in tournament
    completed_matches,      -- Matches finished
    remaining_matches,      -- Matches yet to be played
    total_goals,            -- Total goals scored
    avg_goals_per_match,    -- Average goals per completed match
    current_round,          -- e.g., "Group Stage", "Quarter Finals", "Semi Finals"
    teams_remaining,        -- Number of teams still in competition
    top_scorer_name,        -- Current leading scorer
    top_scorer_goals        -- Goals by top scorer
FROM singhal.fifa_worldcup_gold.gold_tournament_summary;
    """
}

# Execute each query and export to JSON
for req_name, query in queries.items():
    print(f"\n⏳ Processing {req_name}...")
    
    # Execute query
    df = spark.sql(query)
    
    # Convert to pandas and then to dict
    data = df.toPandas().to_dict(orient='records')
    
    # Write JSON file
    json_path = f"{output_dir}/{req_name}.json"
    with open(json_path, 'w') as f:
        json.dump(data, f, indent=2, default=str)
    
    print(f"✅ {req_name}.json - Exported {len(data)} records")

print("\n" + "="*60)
print("🎉 ALL 7 REQUIREMENTS EXPORTED SUCCESSFULLY!")
print("="*60)

In [0]:
%python
# List all exported files
import os

print("\n📂 Files in output directory:")
print("="*60)

for filename in sorted(os.listdir(output_dir)):
    filepath = os.path.join(output_dir, filename)
    size = os.path.getsize(filepath)
    print(f"  ✓ {filename:15} ({size:,} bytes)")

print("="*60)
print(f"\n📍 Location: {output_dir}")